# G1_02 — Dati che si difendono

> **Corso ITS D.E.Mo.S. — Laboratorio API + Frontend — Giornata 1**
> Esegui le celle dall'alto verso il basso con **Shift+Invio**. Prima di ogni cella c'è scritto **cosa aspettarti**.
> Se qualcosa non torna, alza la mano: l'errore si legge insieme.

## Cosa faremo (45 minuti)

Nel notebook precedente l'API ha risposto `422` a un titolo troppo corto. **Chi ha fatto quel controllo?**
Una libreria che si chiama **Pydantic**. Oggi pomeriggio la userai dentro FastAPI; adesso la vediamo da sola.

La regola che ci portiamo a casa: **mai fidarsi dei dati che arrivano da fuori.** Chiunque può mandare
qualsiasi cosa alla tua API: un utente distratto, uno script rotto, o qualcuno che vuole farti danno.

### Cella 1 — Il problema: un dizionario accetta tutto

Con un dizionario Python normale non c'è nessun controllo. Guarda cosa passa.

**Cosa aspettarti:** nessun errore. Il "ticket" viene creato anche se non ha senso.

In [ ]:
ticket = {
    "title": "",                 # titolo vuoto
    "description": 12345,        # un numero invece di un testo
    "status": "boh",             # uno stato inventato
}

print("Creato senza problemi:", ticket)
print("Python non si lamenta. Il problema esplode dopo, da qualche altra parte.")

### Cella 2 — La soluzione: un modello Pydantic

Un **modello** descrive la forma corretta dei dati: quali campi, di che tipo, con quali limiti.
Si scrive come una classe che eredita da `BaseModel`.

**Cosa aspettarti:** la versione di Pydantic e un oggetto `TicketIn` creato correttamente.

In [ ]:
import pydantic
from pydantic import BaseModel, Field
from typing import Literal

print("Pydantic versione", pydantic.VERSION)


class TicketIn(BaseModel):
    # Ogni riga: nome_campo: tipo = regole
    title: str = Field(min_length=3, max_length=100)
    description: str = Field(default="", max_length=1000)
    status: Literal["aperto", "in_lavorazione", "chiuso"] = "aperto"


good = TicketIn(title="Stampante del secondo piano non stampa", description="Coda ferma")
print(good)

### Cella 3 — I dati sbagliati vengono rifiutati

Proviamo a creare lo stesso ticket "senza senso" della Cella 1, ma passando dal modello.

**Cosa aspettarti:** un `ValidationError` con **3 errori**, uno per campo. Leggili: dicono esattamente cosa non va.

In [ ]:
from pydantic import ValidationError

try:
    bad = TicketIn(title="", description=12345, status="boh")
    print("Non dovremmo arrivare qui")
except ValidationError as error:
    print("Rifiutato! Errori trovati:", error.error_count())
    print()
    print(error)

### Cella 4 — L'errore in formato JSON

FastAPI, quando rifiuta una richiesta, manda al client **questo stesso errore** in JSON. È il `422` che hai visto nel notebook precedente.
Ogni voce ha `loc` (dove), `msg` (cosa), `type` (categoria).

**Cosa aspettarti:** una lista di 3 dizionari.

In [ ]:
import json

try:
    TicketIn(title="", description=12345, status="boh")
except ValidationError as error:
    for problem in error.errors():
        print(f"campo {problem['loc']}: {problem['msg']}")

### Cella 5 — Da dizionario a modello (ed è quello che fa l'API)

Nella vita reale i dati arrivano come **dizionario** (dal JSON di una richiesta). Si convalidano con `TicketIn(**data)`
oppure `TicketIn.model_validate(data)`. Se passano, hai un oggetto pulito con i campi al posto giusto.

**Cosa aspettarti:** il primo passa (con `status` riempito col valore di default), il secondo viene rifiutato.

In [ ]:
incoming_ok = {"title": "Monitor che sfarfalla", "description": "Postazione 14"}
incoming_bad = {"title": "Ok", "status": "chiuso"}      # titolo di 2 caratteri

ticket = TicketIn.model_validate(incoming_ok)
print("Passa:", ticket)
print("Lo status non c'era nel dizionario, il modello ha messo il default:", ticket.status)
print()

try:
    TicketIn.model_validate(incoming_bad)
except ValidationError as error:
    print("Rifiutato:", error.errors()[0]["msg"])

### Cella 6 — Il modello converte quando può, rifiuta quando non può

Pydantic è ragionevole: `"5"` può diventare `5`, ma `"cinque"` no. Vediamolo con un modello per il filtro dell'API.

**Cosa aspettarti:** il primo caso converte (`str` → `int`), il secondo fallisce.

In [ ]:
class TicketQuery(BaseModel):
    limit: int = Field(default=10, ge=1, le=100)   # ge = >=, le = <=


print(TicketQuery(limit="5"))          # stringa che sembra un numero: convertita

try:
    TicketQuery(limit="cinque")
except ValidationError as error:
    print("Rifiutato:", error.errors()[0]["msg"])

try:
    TicketQuery(limit=500)
except ValidationError as error:
    print("Rifiutato:", error.errors()[0]["msg"])

### Cella 7 — Dal modello al JSON (la strada del ritorno)

Quando l'API **risponde**, fa il contrario: da oggetto a dizionario a JSON. Si usa `model_dump()` e `model_dump_json()`.

**Cosa aspettarti:** un dizionario e poi lo stesso contenuto come testo JSON.

In [ ]:
ticket = TicketIn(title="Richiesta nuovo mouse")

print("Dizionario:", ticket.model_dump())
print("JSON:      ", ticket.model_dump_json())

### Cella 8 — Tocca a te

Completa il modello `UserIn` in modo che:
- `username` sia obbligatorio, tra 3 e 20 caratteri;
- `age` sia un intero tra 16 e 120;
- `role` possa essere solo `"studente"` o `"docente"`, con default `"studente"`.

Poi esegui: le due righe con `print` devono passare, la terza deve essere rifiutata.

In [ ]:
class UserIn(BaseModel):
    username: str = Field(min_length=3, max_length=20)
    age: int = Field(...)          # <- completa: ge=16, le=120
    role: str = "studente"         # <- completa: usa Literal come in TicketIn


print(UserIn(username="mario", age=19))
print(UserIn(username="anna", age=30, role="docente"))

try:
    UserIn(username="x", age=12, role="hacker")
    print("Ops: doveva essere rifiutato")
except ValidationError as error:
    print("Rifiutato, errori:", error.error_count())

## Riepilogo

- Un dizionario accetta tutto. Un **modello Pydantic** accetta solo dati con la forma giusta.
- `Field(min_length=..., max_length=..., ge=..., le=...)` mette i limiti. `Literal[...]` limita a valori fissi.
- Dati sbagliati → `ValidationError`, che FastAPI trasforma nel `422` per il client.
- `model_validate(dict)` per entrare, `model_dump()` per uscire.
- **Validare all'ingresso** è la prima difesa di un'API. Non l'unica: la seconda la vediamo nel prossimo notebook.